In [ ]:
from collections import defaultdict


def stein_loeschen(pos, spieler):
    """
    Entfernt einen Stein vom Spielbrett und aktualisiert die Quad-Zählungen.

    Parameters
    ----------
    pos : tuple
        Position des Steins als (Spalte, Zeile).
    spieler : bool
        Spieler, dessen Stein entfernt wird (True = Rot, False = Gelb).

    Returns
    -------
    None
        Der Stein wird aus dem Spielbrett gelöscht und die Zählung
        der betroffenen Vierergruppen (Quads) angepasst.
    """

    del spielbrett[pos]
    for i in quads_indices[pos]:
        quads[i][spieler] -=1

def bewerten():
    """
    Bewertet den den Zustand des Spielbretts.

    Die Funktion iteriert über alle Positionen im Spielbrett und
    überprüft die zugehörigen Vierergruppen (Quads). Gruppen, die
    sowohl gelbe als auch rote Steine enthalten, werden ignoriert.
    Für verbleibende Gruppen wird der Score proportional zur Anzahl
    der Steine einer Farbe berechnet.

    Returns
    -------
    int
        Gesamtbewertung des Spielbretts.
        Positive Werte begünstigen Rot, negative Werte Gelb.

    """

    score = 0
    for pos in spielbrett:
        for i in quads_indices[pos]:
            gelbe, rote = quads[i]
        if gelbe > 0 and rote > 0: continue
        score += rote*10
        score -= gelbe*10
    return score

def zug_liste():
    """
    Ermittelt alle möglichen gültigen Züge im aktuellen Spielzustand.

    Returns
    -------
    list of tuple
        Liste aller möglichen Züge als (Spalte, Zeile)-Tupel, in denen
        ein Stein gesetzt werden kann.
    """

    zuege = []
    for spalte in range(SPALTEN):
        if not spalte_ist_gueltig(spalte): continue
        zeile = finde_tiefste_zeile(spalte)
        zuege.append((spalte,zeile))
    return zuege

def spieler_mensch(spieler):
     """
    Führt einen Zug eines menschlichen Spielers aus.

    Der Spieler gibt eine Spalte ein. Wenn der Zug gültig ist, wird
    der Stein in der tiefsten freien Zeile dieser Spalte gesetzt.

    Parameters
    ----------
    spieler : bool
        Aktiver Spieler (True = Rot, False = Gelb).

    Returns
    -------
    bool
        True, wenn der Zug zu einem Gewinn geführt hat, sonst False.
    """

    while True:
        spalte = int(input('Ihr Zug (Spalte 0-6): '))
        if spalte_ist_gueltig(spalte):
            break
    zeile = finde_tiefste_zeile(spalte)
    win = stein_setzen((spalte,zeile), spieler)
    return win

def spieler_computer(spieler):
    """
    Bestimmt und führt den besten Zug für den Computer aus.

    Alle möglichen Züge werden mit dem Minimax-Algorithmus mit
    Alpha-Beta-Pruning bewertet. Der Zug mit der besten Bewertung
    wird ausgeführt.

    Parameters
    ----------
    spieler : bool
        Aktiver Spieler (True = Rot, False = Gelb).

    Returns
    -------
    bool
        True, wenn der ausgeführte Zug zum Gewinn führt, sonst False.
    """

    bewertete_zuege = []
    for zug in zug_liste():
        win = stein_setzen(zug, spieler)
        score = min_max(7, -999999, 999999, spieler, win)
        stein_loeschen(zug, spieler)
        bewertete_zuege.append((score,zug))

    bewertete_zuege.sort(reverse=spieler)
    score, bester_zug = bewertete_zuege[0]
    win = stein_setzen(bester_zug, spieler)

    print(bewertete_zuege)
    print(f'Spieler {1 if spieler else 2} setzt {bester_zug} mit der Bewertung {score}')
    return win

# rekursiver Alpha-Beta-Suchalgorithmus
def min_max(tiefe, alpha, beta, spieler, win):
    """
    Bewertet Spielzustände mit dem Minimax-Algorithmus und Alpha-Beta-Pruning.

    Der Algorithmus durchsucht rekursiv mögliche Spielzüge bis zu einer
    bestimmten Tiefe und verwendet Alpha-Beta-Pruning, um unnötige
    Berechnungen zu vermeiden.

    Parameters
    ----------
    tiefe : int
        Maximale verbleibende Suchtiefe.
    alpha : int
        Aktueller Alpha-Wert für das Pruning (beste Bewertung des Maximierers).
    beta : int
        Aktueller Beta-Wert für das Pruning (beste Bewertung des Minimierers).
    spieler : bool
        Aktiver Spieler (True = Rot/Maximierer, False = Gelb/Minimierer).
    win : bool
        Gibt an, ob der vorherige Zug bereits zum Gewinn geführt hat.

    Returns
    -------
    int
        Bewertungswert der aktuellen Spielsituation.
    """

    if win:
        return 99999+tiefe if spieler else -99999-tiefe
    if tiefe == 0 or len(spielbrett) == ZELLEN:
        return bewerten()
    spieler = not spieler
    value = -999999 if spieler else 999999
    for zug in zug_liste():
        win = stein_setzen(zug, spieler)
        score = min_max(tiefe-1, alpha, beta, spieler, win)
        stein_loeschen(zug, spieler)
        if spieler:
            value = max(value, score)
            alpha = max(value, alpha)
        else:
            value = min(value, score)
            beta = min(value, beta)
       if alpha >= beta:
           break
    return value

quads = quads_bestimmen()

spieler = True
while True:
    print_spielbrett()
    if spieler:
        win = spieler_mensch(spieler)
    else:
        win = spieler_computer(spieler)
    if win:
        print_spielbrett()
        print('GEWONNEN!')
        break
    spieler = not spieler
